# Laboratorio: Diseñando Combos para "Café COVA" ☕

**Contexto:**
La gerencia de la cafetería de la facultad quiere aumentar sus ingresos. Saben que los estudiantes compran cosas en grupo, pero no saben exactamente qué. Tu trabajo como Ingeniero de Datos es leer los registros crudos de la caja registradora, transformar los datos a una matriz transaccional y aplicar el algoritmo Apriori para encontrar los "Combos" más rentables.

In [7]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# FASE 1: EXTRACCIÓN (Datos de producción)
# El sistema punto de venta (POS) exporta los tickets como una lista de listas.
tickets_cafeteria = [
    ['Cafe Americano', 'Croissant', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Agua'],
    ['Cafe Americano', 'Croissant', 'Jugo de Naranja'],
    ['Te Verde', 'Galletas', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Galletas'],
    ['Cafe Americano', 'Croissant', 'Agua', 'Galletas'],
    ['Frappe Mocha', 'Brownie', 'Agua'],
    ['Te Verde', 'Ensalada', 'Agua'],
    ['Cafe Americano', 'Brownie', 'Jugo de Naranja'],
    ['Frappe Mocha', 'Muffin', 'Brownie']
]

### Reto 1: Ingeniería de Características (De Lista a Matriz)
El algoritmo Apriori no entiende listas de texto. Necesitas convertir `tickets_cafeteria` en un DataFrame de Pandas donde cada columna sea un producto y cada fila (ticket) tenga un `True` o `False` (One-Hot Encoding transaccional).

In [15]:
# TU CÓDIGO AQUÍ:
# Utiliza TransactionEncoder para crear df_cafe_bool
te = TransactionEncoder()
df_cafe_bool = te.fit_transform(tickets_cafeteria)
df_cafe_bool = pd.DataFrame(df_cafe_bool, columns=te.columns_)
df_cafe_bool.head()

,Agua,Brownie,Cafe Americano,Croissant,Ensalada,Frappe Mocha,Galletas,Jugo de Naranja,Muffin,Te Verde
0,True,False,True,True,False,False,False,False,False,False
1,True,False,False,False,False,True,False,False,True,False
2,False,False,True,True,False,False,False,True,False,False
3,True,False,False,False,False,False,True,False,False,True
4,False,False,False,False,False,True,True,False,True,False


### Reto 2: Extracción de Patrones Frecuentes
La gerencia te pide descartar las coincidencias raras. Extrae únicamente los `itemsets` (combinaciones de productos) que aparezcan en **al menos el 25% (0.25)** de los tickets.

In [18]:
# TU CÓDIGO AQUÍ:
itemsets_cafe = apriori(df_cafe_bool, min_support=0.2, use_colnames=True)
print(itemsets_cafe)

    support                                      itemsets
0       0.6                             frozenset({Agua})
1       0.3                          frozenset({Brownie})
2       0.4                   frozenset({Cafe Americano})
3       0.3                        frozenset({Croissant})
4       0.4                     frozenset({Frappe Mocha})
5       0.3                         frozenset({Galletas})
6       0.2                  frozenset({Jugo de Naranja})
7       0.3                           frozenset({Muffin})
8       0.2                         frozenset({Te Verde})
9       0.2             frozenset({Agua, Cafe Americano})
10      0.2                  frozenset({Agua, Croissant})
11      0.2               frozenset({Agua, Frappe Mocha})
12      0.2                   frozenset({Agua, Galletas})
13      0.2                   frozenset({Agua, Te Verde})
14      0.2            frozenset({Frappe Mocha, Brownie})
15      0.3        frozenset({Cafe Americano, Croissant})
16      0.2  f

### Reto 3: Creación de Reglas de Asociación
A partir de los itemsets frecuentes, genera las reglas de asociación. Filtra el resultado para mostrar **únicamente** aquellas reglas que tengan una `Confianza` superior al 60% (0.6).

In [21]:
# TU CÓDIGO AQUÍ:
# Usa association_rules() y guarda el resultado en reglas_cafe
reglas_cafe = association_rules(itemsets_cafe, metric="confidence", min_threshold=0.5)
# Filtra las columnas para imprimir solo: antecedents, consequents, support, confidence, 
reglas_cafe = reglas_cafe[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
print(reglas_cafe)

                               antecedents                        consequents  \
0              frozenset({Cafe Americano})                  frozenset({Agua})   
1                   frozenset({Croissant})                  frozenset({Agua})   
2                frozenset({Frappe Mocha})                  frozenset({Agua})   
3                    frozenset({Galletas})                  frozenset({Agua})   
4                    frozenset({Te Verde})                  frozenset({Agua})   
5                frozenset({Frappe Mocha})               frozenset({Brownie})   
6                     frozenset({Brownie})          frozenset({Frappe Mocha})   
7              frozenset({Cafe Americano})             frozenset({Croissant})   
8                   frozenset({Croissant})        frozenset({Cafe Americano})   
9             frozenset({Jugo de Naranja})        frozenset({Cafe Americano})   
10             frozenset({Cafe Americano})       frozenset({Jugo de Naranja})   
11               frozenset({

### 🧠 Reto 4: Defensa de Ingeniería (Análisis de Negocio)

Revisa la tabla resultante de tus reglas y responde las siguientes preguntas para el gerente de la cafetería:

1. **La regla de mayor Confianza:** Identifica la regla que tiene el mayor nivel de Confianza. Traduce esa regla al español simple (ej. "Si un alumno compra X, existe un Y% de probabilidad de que también compre Z").
2. **El engaño del Soporte vs Lift:** ¿Qué producto de la cafetería se vende muchísimo (alto soporte), pero no deberíamos ponerlo en promoción (combo) con el Frappe Mocha porque su Lift es bajo o cercano a 1? Explica matemáticamente por qué.

# Respuestas:
1. La regla de mayor Confianza es Frappe Mocha
Si un cliente compra Frappe Mocha, existe un 100% de probabilidad de que también compre Muffin.
2. El producto con alto soporte pero bajo Lift es el Agua. Mucha gente compra agua el Lift cercano a 1 indica que la compra de Agua no significa mucho con la compra de Frappe Mocha. Un Lift cercano a 1 significa que la probabilidad de comprar Agua dado que se compra Frappe Mocha es casi igual a solo comprar agua y ya.
